# 06.04 · Context engineering

**Chapter:** Prompt & Context Engineering · **Milestone:** M2

## Goals
- [ ] See the context window as a budget: instructions, examples, retrieved data, tool definitions, history, output
- [ ] Run a needle-in-a-haystack test: does the model find a fact at different context sizes and positions?
- [ ] Catch silent truncation (local servers' context limits)
- [ ] Compact long conversations by summarisation; pack retrieved chunks under a token budget

## Resources
- [Anthropic – Effective context engineering for AI agents](https://www.anthropic.com/engineering/effective-context-engineering-for-ai-agents) — free
- [Lost in the Middle (paper)](https://arxiv.org/abs/2307.03172) — free
- [Chroma – Context Rot research](https://research.trychroma.com/context-rot) — free

**Prompt engineering** = writing good instructions. **Context engineering** = deciding *everything* that goes into the window on
each call — which is what actually determines quality, cost and latency in RAG systems and agents.

In [ ]:
import random
import re
import urllib.request

import httpx
import tiktoken

from aieng import DATA
from aieng.llm import client

enc = tiktoken.get_encoding("o200k_base")
ntok = lambda s: len(enc.encode(s))
c, MODEL = client()
LOCAL = c.base_url.host == "localhost"

## 1. Anatomy of a context window
A typical agent/RAG call, say with a 32k window:

| Part | Typical tokens | Notes |
|---|---|---|
| System prompt (role, rules, format) | 300–3,000 | stable → put first, cacheable (05.05) |
| Tool definitions | 100–300 per tool | 30 tools ≈ 6k tokens before the user says anything |
| Few-shot examples | 200–2,000 | |
| Retrieved documents | 1,000–20,000 | the biggest lever: relevance > quantity |
| Conversation / agent history | grows every turn | must be compacted |
| Reserved for the output | max_tokens | don't forget it |

More context is **not** free: cost scales with input tokens, latency grows, and quality often *drops* as irrelevant text
piles up ("context rot").

## 2. Needle in a haystack
Hide one fact in filler text at different depths and lengths; ask for it. Filler: tiny Shakespeare (downloaded in 04.03).

In [ ]:
path = DATA / "tinyshakespeare.txt"
if not path.exists():
    urllib.request.urlretrieve("https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt", path)
filler_tokens = enc.encode(path.read_text())

NEEDLE = "The secret code for the vault is 7319."
QUESTION = "What is the secret code for the vault? Reply with the number only."


def haystack(n_tokens, depth):
    """depth 0.0 = needle at the start, 1.0 = at the end."""
    start = random.Random(n_tokens).randint(0, len(filler_tokens) - n_tokens)
    filler = enc.decode(filler_tokens[start : start + n_tokens])
    cut = int(len(filler) * depth)
    return filler[:cut] + f"\n{NEEDLE}\n" + filler[cut:]


def ask_native(prompt, num_ctx):
    """Ollama native API so we control num_ctx and see prompt_eval_count (how many tokens the model actually read)."""
    r = httpx.post("http://localhost:11434/api/chat", timeout=600, json={
        "model": MODEL, "stream": False, "think": False, "options": {"num_ctx": num_ctx, "temperature": 0},
        "messages": [{"role": "user", "content": prompt}]}).json()
    return r["message"]["content"], r["prompt_eval_count"]


def ask_api(prompt):
    r = c.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": prompt}], temperature=0)
    return r.choices[0].message.content, r.usage.prompt_tokens


print(f"{'tokens':>7} {'depth':>6}  found  tokens_read")
for n in [1_000, 4_000, 12_000]:
    for depth in [0.0, 0.5, 1.0]:
        prompt = haystack(n, depth) + "\n\n" + QUESTION
        answer, read = ask_native(prompt, num_ctx=16_384) if LOCAL else ask_api(prompt)
        print(f"{n:>7} {depth:>6}  {'yes' if '7319' in answer else 'NO ':3}    {read}")

Single-needle retrieval is the *easy* case; most modern models pass it. Harder real-world variants (several facts, reasoning across
them, distractors that look similar) degrade much sooner. Test your own task, at your real context sizes.

## 3. Silent truncation
Local servers have a configured context length. Ollama truncates the *beginning* of the prompt when it doesn't fit — no error.
A RAG app that "can't find" documents may simply never have seen them.

In [ ]:
if LOCAL:
    prompt = haystack(6_000, 0.0) + "\n\n" + QUESTION  # needle at the very start
    for num_ctx in [2_048, 8_192]:
        answer, read = ask_native(prompt, num_ctx)
        print(f"num_ctx={num_ctx:5}: model read {read} of ~{ntok(prompt)} tokens -> {'found' if '7319' in answer else 'NOT found'}: {answer.strip()[:60]!r}")
else:
    print("hosted APIs return an error instead of truncating — still check your token counts before sending")

Fix: set `num_ctx` (native API / Modelfile) or `OLLAMA_CONTEXT_LENGTH` for the server, and **count tokens before sending**.

## 4. Packing retrieved chunks under a budget
Given chunks with relevance scores, keep the best ones that fit. Put the most relevant at the **start and end** (models attend
best there — "lost in the middle").

In [ ]:
def pack(chunks, budget_tokens):
    """chunks: list of (score, text). Greedy by score, then order best-first/best-last."""
    chosen, used = [], 0
    for score, text in sorted(chunks, reverse=True):
        t = ntok(text)
        if used + t <= budget_tokens:
            chosen.append((score, text)); used += t
    # interleave: 1st best at start, 2nd best at end, 3rd after 1st, ...
    front, back = chosen[0::2], chosen[1::2]
    return [t for _, t in front] + [t for _, t in reversed(back)], used


chunks = [(random.random(), f"[doc {i}] " + enc.decode(filler_tokens[i * 300 : i * 300 + random.randint(80, 400)])) for i in range(20)]
packed, used = pack(chunks, budget_tokens=2_000)
print(f"kept {len(packed)} of {len(chunks)} chunks, {used} tokens:", [re.match(r"\[doc \d+\]", p).group() for p in packed])

## 5. Compacting conversation history
When history exceeds a budget, replace old turns with a summary and keep recent turns verbatim. Agents do the same with tool results
(keep the conclusion, drop the 5,000-token raw output).

In [ ]:
EXTRA = {"reasoning_effort": "none"} if LOCAL else {}


def compact(messages, budget_tokens=300, keep_last=4):
    system, turns = messages[0], messages[1:]
    if sum(ntok(m["content"]) for m in turns) <= budget_tokens or len(turns) <= keep_last:
        return messages
    old, recent = turns[:-keep_last], turns[-keep_last:]
    transcript = "\n".join(f"{m['role']}: {m['content']}" for m in old)
    summary = c.chat.completions.create(model=MODEL, temperature=0, **EXTRA, messages=[{"role": "user", "content":
        "Summarise this conversation in <=80 words. Keep names, decisions, numbers and open questions.\n\n" + transcript}]).choices[0].message.content
    return [system, {"role": "user", "content": f"[Summary of earlier conversation]\n{summary}"}, {"role": "assistant", "content": "Understood."}, *recent]


history = [{"role": "system", "content": "You are a trip-planning assistant."}]
for user, bot in [
    ("I'm Ana, planning 10 days in Peru in July with a $2,500 budget.", "Great! July is dry season in the Andes. " * 8),
    ("I want Machu Picchu and Lake Titicaca, but I'm afraid of altitude sickness.", "Acclimatise in the Sacred Valley first. " * 10),
    ("Also add 2 days of beach at the end. I'm vegetarian.", "Paracas or Máncora work; both have vegetarian options. " * 8),
    ("Which airport should I fly into?", "Lima (LIM), then domestic to Cusco. " * 6),
    ("What about trains to Aguas Calientes?", "PeruRail or Inca Rail from Ollantaytambo. " * 6),
]:
    history += [{"role": "user", "content": user}, {"role": "assistant", "content": bot}]

compacted = compact(history)
print(f"{sum(ntok(m['content']) for m in history)} tokens -> {sum(ntok(m['content']) for m in compacted)} tokens\n")
print(compacted[1]["content"])

Check the summary kept what matters (name, budget, dates, altitude worry, vegetarian). Lossy compaction is a classic source of
"the agent forgot X" bugs — evaluate it like any other component.

## Exercises
1. Harder haystack: hide **three** facts at different depths and ask a question that needs all of them. Plot success vs context size.
2. Use `pack` with real scores: embed chunks and the question with any embedding model (preview of chapter 07) and rerun the needle test with
   retrieval + packing instead of the whole haystack. Compare tokens used and accuracy.
3. Make `compact` keep a running summary (update the previous summary instead of re-summarising everything). Test it over 30 turns.

In [ ]:
# your code here